In [8]:
# Step 1: Load Necessary Libraries
import numpy as np
import os
import pandas as pd
import datetime
import gc
import io
import json
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import networkx as nx
from rdkit.Chem import rdmolops
from rdkit.Chem.rdmolops import AddHs
from rdkit.Chem.rdmolops import GetMolFrags
from rdkit import Chem, RDLogger
from rdkit import DataStructs
from rdkit.Chem import AllChem, Draw, Descriptors
from rdkit.Chem import AtomValenceException
from rdkit.Chem import Descriptors
from rdkit.Chem import MolFromSmiles
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem import rdmolfiles
from rdkit.Chem.Draw import IPythonConsole, MolsToGridImage
from rdkit.Chem.Fingerprints import FingerprintMols
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.DataStructs.cDataStructs import TanimotoSimilarity
import rdkit.RDLogger as rdl
from tensorflow import keras
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.callbacks import TensorBoard
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.utils import plot_model
import tensorflow as tf
from scipy.spatial.distance import pdist, squareform
import concurrent.futures
from tqdm import tqdm
import base64
from IPython.display import display
from IPython.display import Image
from multiprocessing import Pool, cpu_count
from PIL import Image
import gzip
import pickle
import psutil
import pygraphviz as pgv
import time
RDLogger.DisableLog("rdApp.*")
tf.get_logger().setLevel('ERROR')
logger = rdl.logger()
logger.setLevel(rdl.ERROR)
logger.setLevel(rdl.CRITICAL)

2025-02-23 12:58:46.114468: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-02-23 12:58:46.117339: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-02-23 12:58:46.171732: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-02-23 12:58:47.070521: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [9]:
# load filtered csv or save a new one
csv_path = "../data/non_duplicate_filtered_quinolines_zinc15_50atoms.csv"
filtered_csv_path = "../data/non_duplicate_filtered_quinolines_zinc15_50atoms.csv"

allowed_atoms = {'B', 'P'}
max_atoms = 50

In [10]:
# Load filetered csv file
data = pd.read_csv(filtered_csv_path)

data = data.sample(n=100000, random_state=1, replace=False)
data.reset_index(drop=True, inplace=True)

FileNotFoundError: [Errno 2] No such file or directory: '../data/non_duplicate_filtered_quinolines_zinc15_50atoms.csv'

In [3]:
# Step 3: Define Generator Network (Graph Neural Network-based)
class Generator(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(Generator, self).__init__()
        self.gcn1 = GCNConv(input_dim, hidden_dim)
        self.gcn2 = GCNConv(hidden_dim, output_dim)
        self.activation = nn.ReLU()

    def forward(self, noise_vector):
        x = self.activation(self.gcn1(noise_vector))
        x = self.activation(self.gcn2(x))
        return x  # Output is a molecular graph

In [4]:
# Step 4: Define Discriminator Network (Graph-based validity checker)
class Discriminator(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(Discriminator, self).__init__()
        self.gcn1 = GCNConv(input_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, 1)  # Output single validity score
        self.activation = nn.LeakyReLU(0.2)

    def forward(self, graph):
        x = self.activation(self.gcn1(graph))
        validity_score = self.fc(x.mean(dim=1))  # Wasserstein distance approximation
        return validity_score

In [5]:
# Step 5: Wasserstein Loss with Gradient Penalty
def wasserstein_loss(real_scores, fake_scores):
    return torch.mean(fake_scores) - torch.mean(real_scores)  # WGAN loss

def gradient_penalty(discriminator, real_graphs, fake_graphs):
    epsilon = torch.rand_like(real_graphs)
    interpolated_graphs = epsilon * real_graphs + (1 - epsilon) * fake_graphs
    interpolated_scores = discriminator(interpolated_graphs)
    gradients = torch.autograd.grad(outputs=interpolated_scores, inputs=interpolated_graphs,
                                    grad_outputs=torch.ones_like(interpolated_scores),
                                    create_graph=True, retain_graph=True)[0]
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gradient_penalty

In [6]:
# Step 6: Training Loop
def train_wgan(num_epochs, batch_size, lambda_gp=10):
    # Initialize generator and discriminator
    generator = Generator(input_dim=128, hidden_dim=256, output_dim=128)
    discriminator = Discriminator(input_dim=128, hidden_dim=256)
    
    optimizer_G = optim.Adam(generator.parameters(), lr=0.0001, betas=(0.5, 0.9))
    optimizer_D = optim.Adam(discriminator.parameters(), lr=0.0001, betas=(0.5, 0.9))
    
    for epoch in range(num_epochs):
        for real_graphs in load_zinc15_data().batch(batch_size):
            # Train Discriminator
            fake_graphs = generator(torch.randn(batch_size, 128))  # Generate fake molecular graphs
            real_scores = discriminator(real_graphs)
            fake_scores = discriminator(fake_graphs.detach())

            gp = gradient_penalty(discriminator, real_graphs, fake_graphs)
            d_loss = wasserstein_loss(real_scores, fake_scores) + lambda_gp * gp
            
            optimizer_D.zero_grad()
            d_loss.backward()
            optimizer_D.step()
            
            # Train Generator
            if epoch % 5 == 0:  # Train G every 5 D updates
                fake_scores = discriminator(generator(torch.randn(batch_size, 128)))
                g_loss = -torch.mean(fake_scores)  # Maximize D's loss
                
                optimizer_G.zero_grad()
                g_loss.backward()
                optimizer_G.step()

        print(f"Epoch {epoch+1}, D Loss: {d_loss.item()}, G Loss: {g_loss.item()}")

In [7]:
# Step 7: Run Training
train_wgan(num_epochs=1000, batch_size=64)

NameError: name 'load_dataset' is not defined